In [1]:
import os
import numpy as np
import pandas as pd
import chromadb

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

print("All libraries imported successfully!")

All libraries imported successfully!


In [2]:
# Define dataset folder
DATASET_PATH = "../dataset/documents"

documents = []
metadatas = []
ids = []

# Read all text documents
for i, filename in enumerate(sorted(os.listdir(DATASET_PATH))):

    if filename.endswith(".txt"):

        with open(os.path.join(DATASET_PATH, filename),
                  "r", encoding="utf-8") as file:

            content = file.read()

        documents.append(content)

        metadatas.append({
            "title": filename.replace(".txt", "").replace("document", "Document "),
            "category": "AI and Education",
            "source": filename
        })

        ids.append(f"doc_{i+1}")

# Display results
print("Total documents loaded:", len(documents))

for i in range(len(documents)):
    print("\n", ids[i])
    print("Title:", metadatas[i]["title"])
    print("Content:", documents[i][:100])

Total documents loaded: 10

 doc_1
Title: Document 1
Content: Artificial Intelligence is transforming education by providing personalized learning experiences. AI

 doc_2
Title: Document 10
Content: Responsible Artificial Intelligence in education focuses on fairness, privacy, transparency, and acc

 doc_3
Title: Document 2
Content: Machine Learning is a branch of Artificial Intelligence that enables computers to learn from data. I

 doc_4
Title: Document 3
Content: Deep Learning uses artificial neural networks to process complex information. It supports speech rec

 doc_5
Title: Document 4
Content: Natural Language Processing allows computers to understand and generate human language. Educational 

 doc_6
Title: Document 5
Content: Online Learning provides students with access to educational resources through the internet. Student

 doc_7
Title: Document 6
Content: Student Performance Analysis uses educational data to understand academic progress. Teachers can ana

 doc_8
Title: Docum

In [3]:
# Load Sentence Transformer model
model = SentenceTransformer('all-MiniLM-L6-v2')

print("Sentence Transformer model loaded successfully!")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Sentence Transformer model loaded successfully!


In [4]:
# Generate embeddings for all documents
embeddings = model.encode(
    documents,
    show_progress_bar=True,
    convert_to_numpy=True
)

# Check embedding dimensions
print("Embeddings generated successfully!")
print("Number of embeddings:", len(embeddings))
print("Embedding dimensions:", embeddings.shape)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embeddings generated successfully!
Number of embeddings: 10
Embedding dimensions: (10, 384)


In [5]:
# Save embeddings as a NumPy file
np.save("../dataset/embeddings.npy", embeddings)

print("Embeddings saved successfully!")

Embeddings saved successfully!


In [6]:
# Create persistent ChromaDB client
client = chromadb.PersistentClient(
    path="../chroma_db"
)

# Create collection
collection = client.get_or_create_collection(
    name="ai_education_documents",
    metadata={"hnsw:space": "cosine"}
)

print("ChromaDB collection created successfully!")

ChromaDB collection created successfully!


In [7]:
# Store documents in ChromaDB
collection.upsert(
    ids=ids,
    documents=documents,
    embeddings=embeddings.tolist(),
    metadatas=metadatas
)

print("Documents stored successfully in ChromaDB!")
print("Total records:", collection.count())

Documents stored successfully in ChromaDB!
Total records: 10


In [8]:
# Retrieve all stored documents
stored_data = collection.get(
    include=["documents", "metadatas"]
)

print("Database verification")
print("----------------------")
print("Total documents:", len(stored_data["ids"]))

for i in range(len(stored_data["ids"])):
    print("\nID:", stored_data["ids"][i])
    print("Metadata:", stored_data["metadatas"][i])

Database verification
----------------------
Total documents: 10

ID: doc_1
Metadata: {'source': 'document1.txt', 'category': 'AI and Education', 'title': 'Document 1'}

ID: doc_2
Metadata: {'title': 'Document 10', 'category': 'AI and Education', 'source': 'document10.txt'}

ID: doc_3
Metadata: {'source': 'document2.txt', 'title': 'Document 2', 'category': 'AI and Education'}

ID: doc_4
Metadata: {'category': 'AI and Education', 'source': 'document3.txt', 'title': 'Document 3'}

ID: doc_5
Metadata: {'source': 'document4.txt', 'category': 'AI and Education', 'title': 'Document 4'}

ID: doc_6
Metadata: {'category': 'AI and Education', 'source': 'document5.txt', 'title': 'Document 5'}

ID: doc_7
Metadata: {'title': 'Document 6', 'source': 'document6.txt', 'category': 'AI and Education'}

ID: doc_8
Metadata: {'category': 'AI and Education', 'title': 'Document 7', 'source': 'document7.txt'}

ID: doc_9
Metadata: {'source': 'document8.txt', 'title': 'Document 8', 'category': 'AI and Education

In [9]:
def semantic_search(query, top_k=5):

    # Convert query into embedding
    query_embedding = model.encode(query)

    # Search in ChromaDB
    results = collection.query(
        query_embeddings=[query_embedding.tolist()],
        n_results=top_k
    )

    print("\nUser Query:", query)
    print("\nTop", top_k, "Relevant Documents")
    print("-" * 50)

    for i in range(len(results["ids"][0])):

        print("\nRank:", i + 1)
        print("Document ID:", results["ids"][0][i])
        print("Title:", results["metadatas"][0][i]["title"])
        print("Category:", results["metadatas"][0][i]["category"])
        print("Distance:", round(results["distances"][0][i], 4))
        print("Content:", results["documents"][0][i][:200])

    return results


print("Semantic search function created successfully!")

Semantic search function created successfully!


In [10]:
query = "How does AI help students improve their learning?"

results = semantic_search(query, top_k=5)


User Query: How does AI help students improve their learning?

Top 5 Relevant Documents
--------------------------------------------------

Rank: 1
Document ID: doc_1
Title: Document 1
Category: AI and Education
Distance: 0.337
Content: Artificial Intelligence is transforming education by providing personalized learning experiences. AI systems analyze student performance and recommend learning materials based on individual needs. It 

Rank: 2
Document ID: doc_8
Title: Document 7
Category: AI and Education
Distance: 0.4243
Content: An Intelligent Tutoring System uses Artificial Intelligence to provide personalized guidance to students. It identifies knowledge gaps, provides feedback, and recommends suitable learning activities.

Rank: 3
Document ID: doc_2
Title: Document 10
Category: AI and Education
Distance: 0.4284
Content: Responsible Artificial Intelligence in education focuses on fairness, privacy, transparency, and accountability. AI systems should protect student information and

In [11]:
query2 = "What are the applications of chatbots in education?"

results2 = semantic_search(query2, top_k=5)


User Query: What are the applications of chatbots in education?

Top 5 Relevant Documents
--------------------------------------------------

Rank: 1
Document ID: doc_9
Title: Document 8
Category: AI and Education
Distance: 0.1941
Content: Educational Chatbots use Natural Language Processing to interact with students. They answer academic questions, explain concepts, provide study materials, and offer assistance at any time.

Rank: 2
Document ID: doc_5
Title: Document 4
Category: AI and Education
Distance: 0.2935
Content: Natural Language Processing allows computers to understand and generate human language. Educational applications include chatbots, grammar correction, automatic translation, and question-answering sys

Rank: 3
Document ID: doc_1
Title: Document 1
Category: AI and Education
Distance: 0.5475
Content: Artificial Intelligence is transforming education by providing personalized learning experiences. AI systems analyze student performance and recommend learning materials b

In [12]:
# Create 10 user queries
queries = [
    "How does artificial intelligence improve education?",
    "What is machine learning and how is it used?",
    "Explain deep learning applications.",
    "How does NLP help students?",
    "What are the benefits of online learning?",
    "How can teachers analyze student performance?",
    "What is an intelligent tutoring system?",
    "How are chatbots used in education?",
    "How does data science help educational institutions?",
    "Why is responsible AI important in education?"
]

# Save queries into CSV
query_df = pd.DataFrame({
    "Query_ID": range(1, 11),
    "Query": queries
})

query_df.to_csv("../dataset/queries.csv", index=False)

print("10 queries created and saved successfully!")
display(query_df)

10 queries created and saved successfully!


,Query_ID,Query
0,1,How does artificial intelligence improve educa...
1,2,What is machine learning and how is it used?
2,3,Explain deep learning applications.
3,4,How does NLP help students?
4,5,What are the benefits of online learning?
5,6,How can teachers analyze student performance?
6,7,What is an intelligent tutoring system?
7,8,How are chatbots used in education?
8,9,How does data science help educational institu...
9,10,Why is responsible AI important in education?


In [13]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Create TF-IDF vectors
tfidf = TfidfVectorizer(stop_words="english")

tfidf_matrix = tfidf.fit_transform(documents)

print("TF-IDF keyword vectors created successfully!")
print("Matrix shape:", tfidf_matrix.shape)

TF-IDF keyword vectors created successfully!
Matrix shape: (10, 134)


In [14]:
def keyword_search(query, top_k=5):

    query_vector = tfidf.transform([query])

    scores = cosine_similarity(
        query_vector,
        tfidf_matrix
    ).flatten()

    top_indices = np.argsort(scores)[::-1][:top_k]

    results = []

    for rank, idx in enumerate(top_indices, start=1):

        results.append({
            "Rank": rank,
            "Document_ID": ids[idx],
            "Title": metadatas[idx]["title"],
            "Similarity_Score": round(float(scores[idx]), 4),
            "Content": documents[idx]
        })

    return results


print("Keyword search function created successfully!")

Keyword search function created successfully!


In [15]:
test_query = "How does AI help students improve their learning?"

keyword_results = keyword_search(test_query, top_k=5)

display(pd.DataFrame(keyword_results))

,Rank,Document_ID,Title,Similarity_Score,Content
0,1,doc_1,Document 1,0.3161,Artificial Intelligence is transforming educat...
1,2,doc_6,Document 5,0.1776,Online Learning provides students with access ...
2,3,doc_2,Document 10,0.1635,Responsible Artificial Intelligence in educati...
3,4,doc_10,Document 9,0.1563,Data Science in education involves collecting ...
4,5,doc_8,Document 7,0.1218,An Intelligent Tutoring System uses Artificial...


In [16]:
semantic_results = semantic_search(test_query, top_k=5)

print("\nKEYWORD SEARCH RESULTS")

keyword_results = keyword_search(test_query, top_k=5)

display(pd.DataFrame(keyword_results))


User Query: How does AI help students improve their learning?

Top 5 Relevant Documents
--------------------------------------------------

Rank: 1
Document ID: doc_1
Title: Document 1
Category: AI and Education
Distance: 0.337
Content: Artificial Intelligence is transforming education by providing personalized learning experiences. AI systems analyze student performance and recommend learning materials based on individual needs. It 

Rank: 2
Document ID: doc_8
Title: Document 7
Category: AI and Education
Distance: 0.4243
Content: An Intelligent Tutoring System uses Artificial Intelligence to provide personalized guidance to students. It identifies knowledge gaps, provides feedback, and recommends suitable learning activities.

Rank: 3
Document ID: doc_2
Title: Document 10
Category: AI and Education
Distance: 0.4284
Content: Responsible Artificial Intelligence in education focuses on fairness, privacy, transparency, and accountability. AI systems should protect student information and

,Rank,Document_ID,Title,Similarity_Score,Content
0,1,doc_1,Document 1,0.3161,Artificial Intelligence is transforming educat...
1,2,doc_6,Document 5,0.1776,Online Learning provides students with access ...
2,3,doc_2,Document 10,0.1635,Responsible Artificial Intelligence in educati...
3,4,doc_10,Document 9,0.1563,Data Science in education involves collecting ...
4,5,doc_8,Document 7,0.1218,An Intelligent Tutoring System uses Artificial...


In [17]:
# Store all retrieval results
all_results = []

for query_id, query in enumerate(queries, start=1):

    # Semantic Search
    semantic = collection.query(
        query_embeddings=[
            model.encode(query).tolist()
        ],
        n_results=5
    )

    for rank in range(5):
        all_results.append({
            "Query_ID": query_id,
            "Query": query,
            "Search_Method": "Semantic Search",
            "Rank": rank + 1,
            "Document_ID": semantic["ids"][0][rank],
            "Title": semantic["metadatas"][0][rank]["title"],
            "Similarity_Score": round(
                1 - semantic["distances"][0][rank], 4
            )
        })

    # Keyword Search
    keyword = keyword_search(query, top_k=5)

    for item in keyword:
        all_results.append({
            "Query_ID": query_id,
            "Query": query,
            "Search_Method": "Keyword Search",
            "Rank": item["Rank"],
            "Document_ID": item["Document_ID"],
            "Title": item["Title"],
            "Similarity_Score": item["Similarity_Score"]
        })

print("All queries processed successfully!")
print("Total retrieval records:", len(all_results))

All queries processed successfully!
Total retrieval records: 100


In [18]:
# Convert results into DataFrame
results_df = pd.DataFrame(all_results)

# Save CSV
results_df.to_csv(
    "../dataset/retrieved_results.csv",
    index=False
)

print("Retrieval results saved successfully!")

display(results_df.head(10))

Retrieval results saved successfully!


,Query_ID,Query,Search_Method,Rank,Document_ID,Title,Similarity_Score
0,1,How does artificial intelligence improve educa...,Semantic Search,1,doc_1,Document 1,0.6627
1,1,How does artificial intelligence improve educa...,Semantic Search,2,doc_2,Document 10,0.6302
2,1,How does artificial intelligence improve educa...,Semantic Search,3,doc_8,Document 7,0.5466
3,1,How does artificial intelligence improve educa...,Semantic Search,4,doc_5,Document 4,0.5052
4,1,How does artificial intelligence improve educa...,Semantic Search,5,doc_3,Document 2,0.4939
5,1,How does artificial intelligence improve educa...,Keyword Search,1,doc_1,Document 1,0.3126
6,1,How does artificial intelligence improve educa...,Keyword Search,2,doc_2,Document 10,0.2253
7,1,How does artificial intelligence improve educa...,Keyword Search,3,doc_3,Document 2,0.2100
8,1,How does artificial intelligence improve educa...,Keyword Search,4,doc_10,Document 9,0.2035
9,1,How does artificial intelligence improve educa...,Keyword Search,5,doc_8,Document 7,0.1567


In [19]:
# Compare average similarity scores
comparison = results_df.groupby(
    "Search_Method"
)["Similarity_Score"].mean().reset_index()

comparison.columns = [
    "Search Method",
    "Average Similarity Score"
]

display(comparison)

,Search Method,Average Similarity Score
0,Keyword Search,0.154478
1,Semantic Search,0.496728


In [20]:
print("FINAL PROJECT FILES")
print("--------------------")

files = [
    "../dataset/queries.csv",
    "../dataset/retrieved_results.csv",
    "../dataset/embeddings.npy"
]

for file in files:
    print(file, ":", "Available" if os.path.exists(file) else "Missing")

print("\nTotal documents:", len(documents))
print("Total queries:", len(queries))
print("Total retrieval results:", len(results_df))
print("ChromaDB records:", collection.count())

FINAL PROJECT FILES
--------------------
../dataset/queries.csv : Available
../dataset/retrieved_results.csv : Available
../dataset/embeddings.npy : Available

Total documents: 10
Total queries: 10
Total retrieval results: 100
ChromaDB records: 10
